<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning/blob/main/Day7/CaseStudy_Early_Wakeup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Case Study : Early Wakeup Health
- Dataset : https://www.kaggle.com/datasets/nalisha/early-wakeup-health-and-lifestyle-dataset

In [ ]:
import pandas as pd
df = pd.read_csv('/content/early_wakeup_health_dataset.csv')

In [ ]:
df.shape

(10000, 64)

In [ ]:
df.isna().sum().sum()

np.int64(4662)

In [ ]:
df.isna().sum()

,0
Person_ID,0
Age,0
Gender,0
Height_cm,0
Weight_kg,0
...,...
Health_Score,0
Fitness_Level,0
Healthy_Aging_Score,0
Wellness_Category,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 64 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Person_ID                       10000 non-null  object 
 1   Age                             10000 non-null  int64  
 2   Gender                          10000 non-null  object 
 3   Height_cm                       10000 non-null  float64
 4   Weight_kg                       10000 non-null  float64
 5   BMI                             10000 non-null  float64
 6   Country                         10000 non-null  object 
 7   Occupation                      10000 non-null  object 
 8   Marital_Status                  10000 non-null  object 
 9   Wake_Up_Time                    10000 non-null  object 
 10  Sleep_Time                      10000 non-null  object 
 11  Sleep_Duration_Hours            10000 non-null  float64
 12  Sleep_Quality_Score             1

## drop columns --> personId, height, Weight, Gym_Member,Earlywaker



### Drop Target Columns -->
 - Obesity_risk, Hypertension, sleepdisorder, Cardiovascular,
 - HealthyAgeing, Wellness, Depression_risk_score, Depression_risk

### Target Columns-->
- Fitness_level --> Multi Class Classification task

- Country --> check distribution --> remove less occurence --> Done --> No need to remove

### -->
* Productivity_Score --> to be discussed


In [ ]:
df['Country'].value_counts()

,count
Country,
USA,1468
India,1206
UK,798
Canada,763
Japan,667
Germany,623
Pakistan,598
Australia,574
Brazil,572


In [ ]:
# Drop Columns(irrelevant & target Columns)
df.drop(columns=['Obesity_Risk',
'Hypertension_Risk',
'Diabetes_Risk',
'Cardiovascular_Risk',
'Sleep_Disorder_Risk',
'Health_Score',
'Healthy_Aging_Score',
'Wellness_Category',
'Early_Waker',
'Depression_Risk_Score',
'Gym_Member',
'Height_cm',
'Weight_kg',
'Person_ID'],inplace=True)



In [ ]:
df.shape

(10000, 50)

## Convert Date Time to Category (Wake Up Time & Sleep Time)
### Preprocessing on Wake up Time & Sleep Time
#### Wake up Time --> Convert to categorical
- 4am - 6am --> Early Morning --> 0
- 6am - 8am --> Morning Time --> 1
- 8am - 10am --> Late --> 2
- 10am - 11am --> Very Late -->3

### Sleep Time --> Convert to categorical
- 6pm - 8pm --> Outlier --> 5
- 8pm - 10pm --> Early --> 0
- 10pm - 12am --> On-time --> 1
- 0am - 3am --> Late --> 2
- 3am - 7am --> Very Late --> 4




In [ ]:
df['Wake_Up_Time'] = pd.to_datetime(df['Wake_Up_Time'])
df['Sleep_Time'] = pd.to_datetime(df['Sleep_Time'])

/tmp/ipykernel_1848/214474565.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Wake_Up_Time'] = pd.to_datetime(df['Wake_Up_Time'])
/tmp/ipykernel_1848/214474565.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Sleep_Time'] = pd.to_datetime(df['Sleep_Time'])


In [ ]:
df['Wake_Up_Time'].head()

,Wake_Up_Time
0,2026-06-27 08:59:00
1,2026-06-27 06:35:00
2,2026-06-27 07:31:00
3,2026-06-27 08:36:00
4,2026-06-27 07:07:00


In [ ]:
df['WT_cat'] = 0 #WakeUp_time -->categorical

In [ ]:
df.loc[(df['Wake_Up_Time'] >= "2026-06-27 04:00:00") & (df['Wake_Up_Time'] < "2026-06-27 06:00:00"), "WT_cat" ] = 0
df.loc[(df['Wake_Up_Time'] >= "2026-06-27 06:00:00") & (df['Wake_Up_Time'] < "2026-06-27 08:00:00"), "WT_cat" ] = 1
df.loc[(df['Wake_Up_Time'] >= "2026-06-27 08:00:00") & (df['Wake_Up_Time'] < "2026-06-27 10:00:00"), "WT_cat" ] = 2
df.loc[(df['Wake_Up_Time'] >= "2026-06-27 10:00:00") & (df['Wake_Up_Time'] <= "2026-06-27 11:00:00"), "WT_cat" ] = 3

In [ ]:
df['WT_cat'].unique()

array([2, 1, 0, 3])

In [ ]:
df['Sleep_Time'].head()

,Sleep_Time
0,2026-06-27 01:29:00
1,2026-06-27 02:25:00
2,2026-06-27 02:24:00
3,2026-06-27 01:19:00
4,2026-06-27 01:20:00


In [ ]:
df['Sleep_Time'].sort_values()

,Sleep_Time
7302,2026-06-27 00:00:00
9058,2026-06-27 00:00:00
7792,2026-06-27 00:00:00
7692,2026-06-27 00:00:00
5933,2026-06-27 00:00:00
...,...
7273,2026-06-27 23:59:00
5927,2026-06-27 23:59:00
7115,2026-06-27 23:59:00
2006,2026-06-27 23:59:00


In [ ]:
df['ST_cat'] = 0

In [ ]:
df.loc[(df['Sleep_Time'] >= "2026-06-27 20:00:00") & (df['Sleep_Time'] < "2026-06-27 22:00:00"), "ST_cat" ] = 0
df.loc[(df['Sleep_Time'] >= "2026-06-27 22:00:00") & (df['Sleep_Time'] <= "2026-06-27 23:59:00"), "ST_cat" ] = 1
df.loc[(df['Sleep_Time'] >= "2026-06-27 00:00:00") & (df['Sleep_Time'] < "2026-06-27 03:00:00"), "ST_cat" ] = 2
df.loc[(df['Sleep_Time'] >= "2026-06-27 03:00:00") & (df['Sleep_Time'] < "2026-06-27 07:00:00"), "ST_cat" ] = 3
## Outlier
df.loc[(df['Sleep_Time'] >= "2026-06-27 18:00:00") & (df['Sleep_Time'] < "2026-06-27 20:00:00"), "ST_cat" ] = 4

In [ ]:
df['ST_cat'].unique()

array([2, 1, 0, 4, 3])

In [ ]:
df.drop(columns=['Wake_Up_Time', 'Sleep_Time'],inplace=True)

In [ ]:
df.shape

(10000, 50)

In [ ]:
df.columns

Index(['Age', 'Gender', 'BMI', 'Country', 'Occupation', 'Marital_Status',
       'Sleep_Duration_Hours', 'Sleep_Quality_Score',
       'Number_of_Night_Awakenings', 'Weekend_Sleep_Difference_Hours',
       'Nap_Frequency_Per_Week', 'Screen_Time_Before_Bed_Hours',
       'Exercise_Frequency_Per_Week', 'Exercise_Duration_Minutes',
       'Exercise_Type', 'Daily_Steps', 'Morning_Workout', 'Workout_Intensity',
       'Daily_Calorie_Intake', 'Water_Intake_Liters', 'Fruit_Intake_Per_Day',
       'Vegetable_Intake_Per_Day', 'Protein_Intake_Grams',
       'Sugary_Drinks_Per_Week', 'Fast_Food_Meals_Per_Week',
       'Breakfast_Regularity_Score', 'Smoking_Status', 'Alcohol_Consumption',
       'Stress_Level', 'Working_Hours_Per_Day', 'Sitting_Hours_Per_Day',
       'Outdoor_Time_Hours', 'Social_Interaction_Score', 'Meditation_Practice',
       'Resting_Heart_Rate', 'Systolic_BP', 'Diastolic_BP',
       'Cholesterol_Level', 'Blood_Sugar_Level', 'Energy_Level_Score',
       'Fatigue_Level_Score', 

## Fill missing Values

In [ ]:
df.isnull().sum() #Exercise_Type,Workout_Intensity,Alcohol_Consumption

,0
Age,0
Gender,0
BMI,0
Country,0
Occupation,0
Marital_Status,0
Sleep_Duration_Hours,0
Sleep_Quality_Score,0
Number_of_Night_Awakenings,0
Weekend_Sleep_Difference_Hours,0


## According to Statisticians ,When we replace it with central tendency then the data is balanced

- Be careful while replacing with mode use .iloc[0] --> in case of multiple modal

In [ ]:
df.loc[df['Alcohol_Consumption'].isna() == True, 'Alcohol_Consumption'] = 0

In [ ]:
df[['Exercise_Type','Workout_Intensity']].mode().iloc[0]

,0
Exercise_Type,Weight Training
Workout_Intensity,Moderate


In [ ]:
#fill_values = df[['Exercise_Type','Workout_Intensity','Alcohol_Consumption']].mode().iloc[0]
#df[['Exercise_Type','Workout_Intensity','Alcohol_Consumption']].fillna(fill_values,inplace=True)

In [ ]:
fill_values = df.mode().iloc[0]
df.fillna(fill_values,inplace=True)

In [ ]:
df.isnull().sum().sum()

np.int64(0)

## Handle Ordinal columns (label_encoding):
- Workout_Intensity
- Smoking_Status
- Alcohol_Consumption

In [ ]:
df['Workout_Intensity'].unique()

array(['Low', 'Moderate', 'High'], dtype=object)

In [ ]:
df['Smoking_Status'].unique()

array(['Former', 'Current', 'Never'], dtype=object)

In [ ]:
df['Alcohol_Consumption'].unique()

array(['Moderate', 0, 'Light', 'Heavy'], dtype=object)

In [ ]:
label_map={
    'Workout_Intensity':{
        'Low':0,
        'Moderate':1,
        'High':2
    },
    'Smoking_Status':{
        'Former':1, 'Current':2, 'Never':0
    },
    'Alcohol_Consumption':{
        'Light':1,
        'Moderate':2,
        'Heavy':3
    }
}

In [ ]:
df.replace(label_map,inplace=True)

/tmp/ipykernel_1848/1715271054.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(label_map,inplace=True)


In [ ]:
df[df.columns[df.nunique() > 15]].dtypes

,0
Age,int64
BMI,float64
Sleep_Duration_Hours,float64
Sleep_Quality_Score,float64
Weekend_Sleep_Difference_Hours,float64
Screen_Time_Before_Bed_Hours,float64
Exercise_Duration_Minutes,int64
Daily_Steps,int64
Daily_Calorie_Intake,int64
Water_Intake_Liters,float64


In [ ]:
df['Workout_Intensity'].unique()

array([0, 1, 2])

In [ ]:
df['Alcohol_Consumption'].unique()

array([2, 0, 1, 3])

In [ ]:
df['Smoking_Status'].unique()

array([1, 2, 0])

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 50 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Age                             10000 non-null  int64  
 1   Gender                          10000 non-null  object 
 2   BMI                             10000 non-null  float64
 3   Country                         10000 non-null  object 
 4   Occupation                      10000 non-null  object 
 5   Marital_Status                  10000 non-null  object 
 6   Sleep_Duration_Hours            10000 non-null  float64
 7   Sleep_Quality_Score             10000 non-null  float64
 8   Number_of_Night_Awakenings      10000 non-null  int64  
 9   Weekend_Sleep_Difference_Hours  10000 non-null  float64
 10  Nap_Frequency_Per_Week          10000 non-null  int64  
 11  Screen_Time_Before_Bed_Hours    10000 non-null  float64
 12  Exercise_Frequency_Per_Week     1

# X & Y Split

- target : fitness level

In [ ]:
X = df.drop(columns=['Fitness_Level'])
Y = df['Fitness_Level']

X.shape, Y.shape

((10000, 49), (10000,))

In [ ]:
Y.unique()

array(['Excellent', 'Good', 'Average', 'Poor'], dtype=object)

In [ ]:
label_map = {'Excellent':3, 'Good':2, 'Average':1, 'Poor':0}
Y.replace(label_map,inplace=True)

/tmp/ipykernel_1848/1394058508.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Y.replace(label_map,inplace=True)


In [ ]:
Y.unique()

array([3, 2, 1, 0])

In [ ]:
X_ohe = pd.get_dummies(X)
X_ohe.shape

(10000, 91)

In [ ]:
X_ohe.columns

Index(['Age', 'BMI', 'Sleep_Duration_Hours', 'Sleep_Quality_Score',
       'Number_of_Night_Awakenings', 'Weekend_Sleep_Difference_Hours',
       'Nap_Frequency_Per_Week', 'Screen_Time_Before_Bed_Hours',
       'Exercise_Frequency_Per_Week', 'Exercise_Duration_Minutes',
       'Daily_Steps', 'Workout_Intensity', 'Daily_Calorie_Intake',
       'Water_Intake_Liters', 'Fruit_Intake_Per_Day',
       'Vegetable_Intake_Per_Day', 'Protein_Intake_Grams',
       'Sugary_Drinks_Per_Week', 'Fast_Food_Meals_Per_Week',
       'Breakfast_Regularity_Score', 'Smoking_Status', 'Alcohol_Consumption',
       'Stress_Level', 'Working_Hours_Per_Day', 'Sitting_Hours_Per_Day',
       'Outdoor_Time_Hours', 'Social_Interaction_Score', 'Resting_Heart_Rate',
       'Systolic_BP', 'Diastolic_BP', 'Cholesterol_Level', 'Blood_Sugar_Level',
       'Energy_Level_Score', 'Fatigue_Level_Score', 'Immune_Health_Score',
       'Mood_Score', 'Anxiety_Score', 'Productivity_Score',
       'Focus_Concentration_Score', 'Life_S

In [ ]:
cat_col = X_ohe.columns[X_ohe.nunique() <= 15]
cat_col.shape

(61,)

In [ ]:
cont_col = X_ohe.columns[X_ohe.nunique() > 15]
cont_col.shape

(30,)

## Train-test Split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X_ohe, Y, test_size=0.3, random_state=7,stratify=Y)
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((7000, 91), (3000, 91), (7000,), (3000,))

In [ ]:
from sklearn.preprocessing import RobustScaler
rc = RobustScaler()
X_train[cont_col] = rc.fit_transform(X_train[cont_col])
X_test[cont_col] = rc.transform(X_test[cont_col])

In [ ]:
X_train[cont_col].shape, X_test[cont_col].shape

((7000, 30), (3000, 30))

## Apply Outlier Imputation Using IQR

In [ ]:
def outlier_impute_iqr(df,col):
  df1 = df.copy()
  print("Column --> ",col)
  q1,q3 = df1[col].quantile([0.25,0.75])
  iqr = q3-q1
  lb = q1 - 1.5*iqr
  ub = q3 + 1.5*iqr
  df1.loc[(df1[col] < lb),col] = lb
  df1.loc[(df1[col] > ub),col] = ub
  #df1.loc[(df1[col]UB),col] = UB
  print(lb, ub, df1[col].min(), df1[col].max())
  return df1

In [ ]:
X_train_io = X_train.copy()
for col in cont_col:
  X_train_io = outlier_impute_iqr(X_train_io,col)

Column -->  Age
-1.9642857142857144 2.0357142857142856 -0.8928571428571429 1.3214285714285714
Column -->  BMI
-2.0089153046062407 1.9910846953937595 -1.4257057949479945 1.9910846953937595
Column -->  Sleep_Duration_Hours
-2.0033557046979866 1.9966442953020134 -2.0033557046979866 1.8523489932885902
Column -->  Sleep_Quality_Score
-2.0 2.0 -2.0 1.7272727272727282
Column -->  Weekend_Sleep_Difference_Hours
-2.0092592592592595 1.9907407407407407 -1.8888888888888888 1.9907407407407407
Column -->  Screen_Time_Before_Bed_Hours
-1.8664122137404582 2.133587786259542 -0.618320610687023 2.133587786259542
Column -->  Exercise_Duration_Minutes
-2.0 2.0 -1.6071428571428572 2.0
Column -->  Daily_Steps
-2.025921583952065 1.9740784160479354 -1.9954409274456169 1.9740784160479354
Column -->  Daily_Calorie_Intake
-1.9973230220107079 2.0026769779892923 -1.9973230220107079 2.0026769779892923
Column -->  Water_Intake_Liters
-1.9999999999999996 2.0000000000000004 -1.9999999999999996 2.0000000000000004
Column

## Feature Selection
-                       X_train            X_train_io
- PCA (0.95)            X_train_PCA        X_train_io_pca
- KPCA ('rbf')          X_train_kpca       X_train_io_kpca
- RFE                   X_train_rfe        X_train_io_rfe
- DT('balanced')        X_train_dt         X_train_io_dt

## PCA

X_train

In [ ]:
from sklearn.decomposition import PCA, KernelPCA
pca = PCA(n_components=0.95,random_state=7)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
X_train_pca.shape , X_test_pca.shape

((7000, 44), (3000, 44))

X_train_io

In [ ]:
X_train_io_pca = pca.fit_transform(X_train_io)
X_test_io_pca = pca.transform(X_test)
X_train_io_pca.shape , X_test_io_pca.shape

((7000, 44), (3000, 44))

## KPCA

- X_train

In [ ]:
kpca = KernelPCA(kernel="rbf",random_state=7)
X_train_kpca = kpca.fit_transform(X_train)
X_test_kpca = kpca.transform(X_test)
X_train_kpca.shape , X_test_kpca.shape

((7000, 6999), (3000, 6999))

In [ ]:
kpca.eigenvalues_

array([4.01043174e+02, 3.62946990e+02, 2.48024846e+02, ...,
       1.26932874e-02, 1.24345785e-02, 1.09835317e-02])

In [ ]:
kpca = KernelPCA(kernel="rbf",random_state=7)
X_train_io_kpca = kpca.fit_transform(X_train_io)
X_test_io_kpca = kpca.transform(X_test)
X_train_io_kpca.shape , X_test_io_kpca.shape

((7000, 6999), (3000, 6999))

## RFE

In [ ]:
#X_train
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
Xtr_rfe = RFE(LogisticRegression(random_state=7,class_weight='balanced'),verbose=2)
Xtr_rfe.fit(X_train,Y_train)
print(Xtr_rfe.ranking_)

Fitting estimator with 91 features.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fitting estimator with 90 features.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fitting estimator with 89 features.


...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fitting estimator with 46 features.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[ 1  1  5  1 42 36 37 46  1 22  1 45  1  1 25  1  9 47  1 31  1 14  1 21
 43 44  1 35 27 32 30 19 28 39  1  1 10 29 40 13  3  4  1  1  1  1 20 18
 16  1  1  8 17 34  7  1  1 23  1  1 11 12 26  1  1  1  1  1  1  1  1  1
  1 41 24  1  6  1 15  1  1  2 33 38  1  1  1  1  1  1  1]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
selected_col = X_train.columns[Xtr_rfe.ranking_ == 1]
selected_col.shape

(45,)

In [ ]:
X_train_rfe = X_train.loc[:,selected_col]
X_test_rfe = X_test.loc[:,selected_col]


In [ ]:
for col,rank in  zip(X_train.columns,Xtr_rfe.ranking_):
  print(col,rank)

Age 1
BMI 1
Sleep_Duration_Hours 5
Sleep_Quality_Score 1
Number_of_Night_Awakenings 42
Weekend_Sleep_Difference_Hours 36
Nap_Frequency_Per_Week 37
Screen_Time_Before_Bed_Hours 46
Exercise_Frequency_Per_Week 1
Exercise_Duration_Minutes 22
Daily_Steps 1
Workout_Intensity 45
Daily_Calorie_Intake 1
Water_Intake_Liters 1
Fruit_Intake_Per_Day 25
Vegetable_Intake_Per_Day 1
Protein_Intake_Grams 9
Sugary_Drinks_Per_Week 47
Fast_Food_Meals_Per_Week 1
Breakfast_Regularity_Score 31
Smoking_Status 1
Alcohol_Consumption 14
Stress_Level 1
Working_Hours_Per_Day 21
Sitting_Hours_Per_Day 43
Outdoor_Time_Hours 44
Social_Interaction_Score 1
Resting_Heart_Rate 35
Systolic_BP 27
Diastolic_BP 32
Cholesterol_Level 30
Blood_Sugar_Level 19
Energy_Level_Score 28
Fatigue_Level_Score 39
Immune_Health_Score 1
Mood_Score 1
Anxiety_Score 10
Productivity_Score 29
Focus_Concentration_Score 40
Life_Satisfaction_Score 13
WT_cat 3
ST_cat 4
Gender_Female 1
Gender_Male 1
Gender_Non-binary 1
Country_Australia 1
Country_Brazi

In [ ]:
#X_train_io
Xtr_io_rfe = RFE(LogisticRegression(random_state=7,class_weight='balanced'),verbose=2)
Xtr_io_rfe.fit(X_train_io,Y_train)
print(Xtr_io_rfe.ranking_)

Fitting estimator with 91 features.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fitting estimator with 90 features.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fitting estimator with 89 features.


...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fitting estimator with 46 features.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[ 1  1  4  1 44 36 40 46  1 28  1 43  1  1 30  1  9 47  1 33  1 10  1 17
 42 41  1 38 25 29 24 13 23 37  1  1 12 26 39  6  2  3  1  1  1  1 32 22
 21  1  1  7 19 34  5  1  1 27  1 18  1  8 16  1  1  1 14  1  1  1  1  1
  1 45 31  1 11  1 15  1  1  1 20 35  1  1  1  1  1  1  1]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
selected_col = X_train_io.columns[Xtr_io_rfe.ranking_ == 1]
selected_col.shape

(45,)

In [ ]:
X_train_io_rfe = X_train_io.loc[:,selected_col]
X_test_io_rfe = X_test.loc[:,selected_col]

In [ ]:
for col,rank in  zip(X_train_io.columns,Xtr_io_rfe.ranking_):
  print(col,rank)

Age 1
BMI 1
Sleep_Duration_Hours 1
Sleep_Quality_Score 1
Number_of_Night_Awakenings 41
Weekend_Sleep_Difference_Hours 43
Nap_Frequency_Per_Week 46
Screen_Time_Before_Bed_Hours 39
Exercise_Frequency_Per_Week 1
Exercise_Duration_Minutes 40
Daily_Steps 1
Workout_Intensity 42
Daily_Calorie_Intake 33
Water_Intake_Liters 1
Fruit_Intake_Per_Day 5
Vegetable_Intake_Per_Day 1
Protein_Intake_Grams 4
Sugary_Drinks_Per_Week 47
Fast_Food_Meals_Per_Week 1
Breakfast_Regularity_Score 31
Smoking_Status 1
Alcohol_Consumption 1
Stress_Level 1
Working_Hours_Per_Day 27
Sitting_Hours_Per_Day 44
Outdoor_Time_Hours 38
Social_Interaction_Score 19
Resting_Heart_Rate 29
Systolic_BP 1
Diastolic_BP 30
Cholesterol_Level 35
Blood_Sugar_Level 34
Energy_Level_Score 20
Fatigue_Level_Score 10
Immune_Health_Score 1
Mood_Score 1
Anxiety_Score 1
Productivity_Score 45
Focus_Concentration_Score 37
Life_Satisfaction_Score 1
WT_cat 1
ST_cat 1
Gender_Female 1
Gender_Male 1
Gender_Non-binary 8
Country_Australia 1
Country_Brazil 2

## Decision Tree

- Xtrain

In [ ]:
from sklearn.tree import DecisionTreeClassifier
Xtr_dt = DecisionTreeClassifier(random_state=7,criterion='gini',class_weight='balanced')
Xtr_dt.fit(X_train,Y_train)

DecisionTreeClassifier(class_weight='balanced', random_state=7)

In [ ]:
fi = zip(X_train.columns,Xtr_dt.feature_importances_)
for col,imp in sorted(fi,key=lambda x:x[1],reverse=True):
  print(col,imp)

Energy_Level_Score 0.20786087152876345
BMI 0.20656000807579594
Daily_Steps 0.05161470183579584
Mood_Score 0.04556149930298373
Age 0.03345588775985051
Fatigue_Level_Score 0.030281064868787698
Sleep_Quality_Score 0.02753877424232547
Stress_Level 0.026761871632850633
Immune_Health_Score 0.023597385592683113
Resting_Heart_Rate 0.023170507090374874
Fast_Food_Meals_Per_Week 0.022324114893128487
Anxiety_Score 0.019763500492052007
Water_Intake_Liters 0.01820836222869746
Protein_Intake_Grams 0.01623274262331617
Blood_Sugar_Level 0.015655508922512388
Exercise_Frequency_Per_Week 0.014977683610784042
Sleep_Duration_Hours 0.012262024354820537
Systolic_BP 0.011616316237202886
Sitting_Hours_Per_Day 0.011554950335908268
Social_Interaction_Score 0.01117779606688283
Productivity_Score 0.011119563494161846
Life_Satisfaction_Score 0.010948562652654198
Outdoor_Time_Hours 0.010585618216022571
Focus_Concentration_Score 0.01042120499540775
Weekend_Sleep_Difference_Hours 0.00952027061526029
Screen_Time_Before_

In [ ]:
selected_col = X_train.columns[Xtr_dt.feature_importances_ > Xtr_dt.feature_importances_.mean()]
selected_col.shape

(21,)

In [ ]:
X_train_dt = X_train.loc[:,selected_col]
X_test_dt = X_test.loc[:,selected_col]

- Xtrain_io

In [ ]:
Xtrio_dt = DecisionTreeClassifier(random_state=7,criterion='gini',class_weight='balanced')
Xtrio_dt.fit(X_train_io,Y_train)

DecisionTreeClassifier(class_weight='balanced', random_state=7)

In [ ]:
fi = zip(X_train_io.columns,Xtrio_dt.feature_importances_)
for col,imp in sorted(fi,key=lambda x:x[1],reverse=True):
  print(col,imp)

Energy_Level_Score 0.2086909269620671
BMI 0.20766265453786079
Daily_Steps 0.05068601262804339
Mood_Score 0.04543695002501746
Age 0.032430519565679425
Fatigue_Level_Score 0.03115385135320318
Sleep_Quality_Score 0.028495335941183475
Stress_Level 0.025356471866568926
Fast_Food_Meals_Per_Week 0.024182450931678087
Resting_Heart_Rate 0.022566113895905956
Immune_Health_Score 0.021860564676089
Anxiety_Score 0.019860854672052453
Blood_Sugar_Level 0.016488515001584714
Sitting_Hours_Per_Day 0.015179443938220516
Exercise_Frequency_Per_Week 0.015004890166018447
Sleep_Duration_Hours 0.013462901288785436
Cholesterol_Level 0.012577812362209955
Life_Satisfaction_Score 0.012440409850383598
Protein_Intake_Grams 0.011629425791759839
Social_Interaction_Score 0.011115063796144615
Water_Intake_Liters 0.010965988181271208
Productivity_Score 0.010823088786563321
Weekend_Sleep_Difference_Hours 0.01028492197894999
Systolic_BP 0.009898660753872563
Focus_Concentration_Score 0.00924549162168672
Exercise_Duration_Mi

In [ ]:
selected_col = X_train_io.columns[Xtrio_dt.feature_importances_ > Xtrio_dt.feature_importances_.mean()]
selected_col.shape

(20,)

In [ ]:
X_train_io_dt = X_train.loc[:,selected_col]
X_test_io_dt = X_test.loc[:,selected_col]

## Apply Models
  * Multi Class Classification
    - Logistic Regression
    - Decision Trees
    - SVM
    - KNN
    - Random Forest
    - XGBoost
    - AdaBoost
    - CatBoost
    - LightGBM


## Best Classificatipn Model Will depend on the dataset will select 3 models

 - Basic ML model(logistic Regression, SVM, KNN)
 - Random Forest (Bagging)
 - XGBoost, CatBoost, LightGBM

# Selected Classifiers for this Dataset

- Logistic Regression
- CatBoost
- Random Forest
-  LightGBM

## Steps to be done

1. Take the Model
2. Train the Model
3. Get the Predictions Y_Predict and Y_Predict_prob
4. Classification Report
5. Log Loss
6. Average Precision Score

In [ ]:
# Imbalance Data
Y.value_counts(normalize=True)

,proportion
Fitness_Level,
2,0.4429
3,0.3320
1,0.2137
0,0.0114


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
lr = LogisticRegression(random_state=7)
lr.fit(X_train, Y_train)
Y_pred = lr.predict(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       0.86      0.53      0.65        34
           1       0.86      0.79      0.82       641
           2       0.81      0.86      0.83      1329
           3       0.88      0.86      0.87       996

    accuracy                           0.84      3000
   macro avg       0.85      0.76      0.79      3000
weighted avg       0.84      0.84      0.84      3000



In [ ]:
lr.fit(X_train_io, Y_train)
Y_pred_io = lr.predict(X_test)
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       0.86      0.53      0.65        34
           1       0.86      0.79      0.82       641
           2       0.81      0.86      0.83      1329
           3       0.88      0.86      0.87       996

    accuracy                           0.84      3000
   macro avg       0.85      0.76      0.79      3000
weighted avg       0.84      0.84      0.84      3000



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.4 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier
cbc = CatBoostClassifier(iterations=100, learning_rate=0.2,depth=5,cat_features=None,
                        random_state=7,verbose=False)
cbc.fit(X_train, Y_train)
y_pred_cbc_xtr = cbc.predict(X_test)

In [ ]:
y_pred_prob_cbc_xtr = cbc.predict_proba(X_test)

In [ ]:
from sklearn.metrics import log_loss, average_precision_score
print("Classifier Catboost and Dataset X_train")
ll = log_loss(Y_test, y_pred_prob_cbc_xtr)
print("Log Loss --> ",ll)
aps = average_precision_score(Y_test, y_pred_prob_cbc_xtr)
print("Average Precision Score --> ",aps)

Classifier Catboost and Dataset X_train
Log Loss -->  0.40453868410220606
Average Precision Score -->  0.849298025888901


In [214]:
from sklearn.metrics import log_loss, average_precision_score, precision_score, recall_score, f1_score
def eval_model(model_name, dataset_name,model,X_train,Y_train,X_test,Y_test):
  model.fit(X_train, Y_train)
  y_pred = model.predict(X_test)
  y_pred_prob = model.predict_proba(X_test)
  precision = precision_score(Y_test, y_pred,average='weighted')
  recall = recall_score(Y_test, y_pred,average='weighted')
  f1 = f1_score(Y_test, y_pred,average='weighted')
  ll = log_loss(Y_test, y_pred_prob)
  aps = average_precision_score(Y_test, y_pred_prob)
  return [model_name, dataset_name, precision, recall, f1, ll, aps]

Create a function which will take model , Xtrain, Xtest, YTrain, Ytest as a Parameter, return  precision, recall, f1score, log loss , average Precision score
run this function on diff datasets and models
store the list in list of Lists
Convert it to a dataframe Compare the results in the Dataframe
Plot the results if Possible

In [215]:
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

datasets = {
    "X_train": (X_train, X_test),
    "X_train_io": (X_train_io, X_test),
    "X_train_pca": (X_train_pca, X_test_pca),
    "X_train_io_pca": (X_train_io_pca, X_test_io_pca),
    #"X_train_kpca": (X_train_kpca, X_test_kpca),
    "X_train_io_kpca": (X_train_io_kpca, X_test_io_kpca),
    "X_train_rfe": (X_train_rfe, X_test_rfe),
    "X_train_io_rfe": (X_train_io_rfe, X_test_io_rfe),
    "X_train_dt": (X_train_dt, X_test_dt),
    "X_train_io_dt": (X_train_io_dt, X_test_io_dt)
}

models={
    "RandomForest": RandomForestClassifier(random_state=7,n_estimators=100,max_depth=10,class_weight='balanced'),
    "LightGBM": LGBMClassifier(max_depth=5,learning_rate=0.2,n_estimators=100,subsample=0.8,colsample_bytree=0.8,
                      random_state=7,class_weight='balanced'),
    "Logistic Regression": LogisticRegression(random_state=7,class_weight='balanced'),
    "CatBoost": CatBoostClassifier(iterations=100, learning_rate=0.2,depth=5,cat_features=None,
                                   random_state=7,verbose=False)
}




In [217]:
# Master List of Lists to store all metrics rows
list_of_lists = []

# Corrected Loop structure
for dataset_name, (X_tr, X_te) in datasets.items():
    for model_name, clf in models.items():
        try:
            # Explicitly pass the correct unpacked variables
            row = eval_model(model_name, dataset_name, clf, X_tr, Y_train, X_te, Y_test)
            list_of_lists.append(row)

            # Print verification tracking just like your custom format
            print(f"Classifier {model_name} and Dataset {dataset_name} evaluated successfully.")

        except Exception as e:
            print(f"Skipping {model_name} on {dataset_name} due to error: {e}")

# Build and view the dataframe right after the loop finishes
columns = ["Model", "Dataset", "Precision", "Recall", "F1-Score", "Log Loss", "Avg Precision Score"]
results_df = pd.DataFrame(list_of_lists, columns=columns)
results_df.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)

Skipping RandomForest on X_train due to error: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Alcohol_Consumption
- Breakfast_Regularity_Score
- Country_Australia
- Country_Brazil
- Country_Canada
- ...

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007536 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4387
[LightGBM] [Info] Number of data points in the train set: 7000, number of used features: 91
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Skipping Logistic Regression on X_train due to error: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Alcohol_Consumption
- Breakfast_Regularity_Score
- Country_Australia
- Country_Brazil
- Country_Canada
- ...

Skipping CatBoost on X_train due to error: catboost/libs/data/model_dataset_compatibility.cpp:81: At position 4 should be feature with name Number_of_Night_Awakenings (found Exercise_Frequency_Per_Week).
Skipping RandomForest on X_train_io due to error: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Alcohol_Consumption
- Breakfast_Regularity_Score
- Country_Australia
- Country_Brazil
- Country_Canada
- ...

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001481 seconds.
You can set `force_row_wise=true` to remove the overhe

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Skipping Logistic Regression on X_train_io due to error: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Alcohol_Consumption
- Breakfast_Regularity_Score
- Country_Australia
- Country_Brazil
- Country_Canada
- ...

Skipping CatBoost on X_train_io due to error: catboost/libs/data/model_dataset_compatibility.cpp:81: At position 4 should be feature with name Number_of_Night_Awakenings (found Exercise_Frequency_Per_Week).


...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classifier Logistic Regression and Dataset X_train_dt evaluated successfully.
Classifier CatBoost and Dataset X_train_dt evaluated successfully.
Classifier RandomForest and Dataset X_train_io_dt evaluated successfully.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002660 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2485
[LightGBM] [Info] Number of data points in the train set: 7000, number of used features: 20
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classifier Logistic Regression and Dataset X_train_io_dt evaluated successfully.
Classifier CatBoost and Dataset X_train_io_dt evaluated successfully.


,Model,Dataset,Precision,Recall,F1-Score,Log Loss,Avg Precision Score
0,CatBoost,X_train_rfe,0.842684,0.838667,0.838300,0.385755,0.856066
1,CatBoost,X_train_io_rfe,0.842326,0.838667,0.838051,0.386906,0.860131
2,Logistic Regression,X_train_io_rfe,0.833367,0.829667,0.830269,0.395310,0.884583
3,Logistic Regression,X_train_rfe,0.829245,0.825333,0.826073,0.393245,0.885379
4,LightGBM,X_train_io_rfe,0.823176,0.823333,0.822740,0.403357,0.818278
5,LightGBM,X_train_rfe,0.820414,0.821000,0.820493,0.413163,0.808643
6,Logistic Regression,X_train_pca,0.818623,0.814000,0.814822,0.426055,0.855610
7,Logistic Regression,X_train_io_pca,0.818081,0.813333,0.814200,0.427421,0.853681
8,CatBoost,X_train_dt,0.818137,0.815333,0.813836,0.418145,0.827249
9,CatBoost,X_train_io_dt,0.816805,0.813667,0.812858,0.425619,0.828281
